# Training model

In [1]:
import pandas as pd 
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('../datasets/new_data_sets/frame_of_colab_filt.csv')

In [3]:
df['tags'] = df['tags'].fillna("")

In [4]:
df.drop(columns='tags',inplace=True)

In [5]:
df

,Unnamed: 0,movieId,title,genres,overview,content
0,0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,"Led by Woody, Andy's toys live happily in his ...",Toy Story (1995) Adventure Animation Children ...
1,1,2,Jumanji (1995),Adventure Children Fantasy,When siblings Judy and Peter discover an encha...,Jumanji (1995) Adventure Children Fantasy fant...
2,2,3,Grumpier Old Men (1995),Comedy Romance,A family wedding reignites the ancient feud be...,Grumpier Old Men (1995) Comedy Romance moldy o...
3,3,4,Waiting to Exhale (1995),Comedy Drama Romance,"Cheated on, mistreated and stepped on, the wom...",Waiting to Exhale (1995) Comedy Drama Romance ...
4,4,5,Father of the Bride Part II (1995),Comedy,Just when George Banks has recovered from his ...,Father of the Bride Part II (1995) Comedy preg...
...,...,...,...,...,...,...
9606,9607,193581,Black Butler: Book of the Atlantic (2017),Action Animation Comedy Fantasy,"Ciel learns of a ""Aurora Society"", that is rum...",Black Butler: Book of the Atlantic (2017) Acti...
9607,9608,193583,No Game No Life: Zero (2017),Animation Comedy Fantasy,Six thousand years before Sora and Shiro were ...,No Game No Life: Zero (2017) Animation Comedy ...
9608,9609,193585,Flint (2017),Drama,A woman deals with the toxic water scandal in ...,Flint (2017) Drama A woman deals with the tox...
9609,9610,193587,Bungo Stray Dogs: Dead Apple (2018),Action Animation,A large scale catastrophe is occurring across ...,Bungo Stray Dogs: Dead Apple (2018) Action Ani...


In [30]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    min_df=2,
    max_features=100000,
    lowercase=True,
    token_pattern=r"(?u)\b[^\W\d_][\w]*\b"
)
vectorizer.fit(df['content'])

,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b[^\\W\\d_][\\w]*\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",100000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None


In [31]:
tfidf_matrix = vectorizer.transform(df['content'])

In [32]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 339867 stored elements and shape (9611, 36189)>

In [72]:
search_title = "xo kitty"

matches = df[
    df["title"].str.contains(
        search_title,
        case=False,
        na=False,
        regex=False
    )
]

titles = matches["title"].to_numpy()

In [77]:
titles

array([], dtype=object)

In [78]:
def get_index(titles,df):
    match = df.index[
    df['title'].str.casefold() == title.casefold()
    ].tolist()
    return match

In [79]:
n_recs = 15
if titles.shape[0] != 0:
    n_titles = titles.shape[0]
else : 
    n_titles = 1
    
rec_per_title = n_recs//n_titles 
for title in titles :
    crr_index = get_index(title,df)
    print(crr_index)
    similartity_matrix = cosine_similarity(
        tfidf_matrix[crr_index],
        tfidf_matrix
    ).ravel() # cosine similarity gives 2D array .ravel() convert it into 1D array
    similartity_matrix[crr_index] = -1 
    recom_movs_idx = similartity_matrix.argsort()[::-1][:10]
    recoms_for_crr_title = df.iloc[recom_movs_idx][
        ['movieId','title']
    ]
    recoms_for_crr_title['similarity'] = similartity_matrix[recom_movs_idx]
    display(recoms_for_crr_title)
    break
    

In [62]:
movie_title = 'Dark Knight, The (2008)'

# below code is doing that gave me the index of rows where df['title] == movie_title 
matches = df.index[
    df['title'].str.casefold() == movie_title.casefold()
].tolist()
if not matches :
    print("movie dont exist")
else : 
    print("movie founded at index : ",matches[0])
    movie_idx = matches[0]
    similartity = cosine_similarity(
    tfidf_matrix[movie_idx],
    tfidf_matrix
).ravel()

similartity[movie_idx] = -1
top_indices = similartity.argsort()[::-1][:10]
recommendations = df.iloc[top_indices][
    ["movieId", "title", "genres"]
].copy()
recommendations['similarity'] = similartity[top_indices]

movie founded at index :  6673


In [80]:
recommendations

,movieId,title,genres,similarity
7708,91529,"Dark Knight Rises, The (2012)",Action Adventure Crime IMAX,0.324620
126,153,Batman Forever (1995),Action Adventure Comedy Crime,0.237759
8008,99813,"Batman: The Dark Knight Returns, Part 2 (2013)",Action Animation,0.222259
8160,104141,Batman: Mystery of the Batwoman (2003),Action Animation Children Crime,0.207801
5607,27311,Batman Beyond: Return of the Joker (2000),Action Animation Crime Sci-Fi Thriller,0.206716
509,592,Batman (1989),Action Crime Thriller,0.201896
7962,98124,"Batman: The Dark Knight Returns, Part 1 (2012)",Action Animation Sci-Fi,0.191937
6778,60979,Batman: Gotham Knight (2008),Action Animation Crime,0.188712
9248,161354,Batman: The Killing Joke (2016),Action Animation Crime Drama,0.174965
7674,90603,Batman: Year One (2011),Action Animation Crime,0.166435


In [ ]:
1575	2116	Lord of the Rings, The (1978)	Adventure Animation Children Fantasy